# 1. Initializations

## 1.1 General imports

In [ ]:
### global
import logging
import pandas as pd
from smartcheck.logger_config import setup_logger
from pathlib import Path
import smartcheck.paths as pth
setup_logger(logging.INFO)
from sqlalchemy import  Table, Column, Integer, String, ForeignKey, MetaData, create_engine, text, inspect

In [ ]:
from IPython.display import FileLink
# Génère un lien pour un fichier local (dans le même dossier ou sous-dossier)
FileLink('mon_fichier.csv')

In [ ]:
def setup_jupysql_sqlite(path_str: str, echo=True):
    uri = f"sqlite:///{Path(path_str).resolve().as_posix()}"
    engine = create_engine(uri, echo=echo)
    connexion = engine.connect()
    print("✅ Connected to:", uri)
    return connexion

In [ ]:
# Base de données chinook existante
chinook_path = pth.get_full_path("smartcheck\\resources\\learning\\chinook.db")
print("Database 'chinook.db' Full Path:", chinook_path)
# Démarrage du moteur SQL lite sur la BDD
chinook_connexion = setup_jupysql_sqlite(chinook_path, echo=False)
# Base de données college vierge
college_path = pth.get_full_path("smartcheck\\resources\\learning\\colleges.db")
print("Database 'college.db' Full Path:", college_path)
# Démarrage du moteur SQL lite sur la BDD
college_connexion = setup_jupysql_sqlite(college_path, echo=False)

# 2. SQL (python)

#### requests that read (dataframe result)

In [ ]:
chinook_inspector = inspect(chinook_connexion)
display(chinook_inspector.get_table_names())
display(chinook_inspector.get_columns(table_name='albums'))
display(chinook_inspector.get_foreign_keys(table_name='albums'))

In [ ]:
df = pd.read_sql("""
SELECT * FROM albums
""", chinook_connexion)
display(df.head())
chinook_connexion.commit()

#### requests that modify (no result expected)
No alter table in SQLite, use of a new table is needed
>1. Create new table
>2. Copy data
>3. Drop old table
>4. Rename new into old

In [ ]:
with college_connexion.begin() as transaction:
    try:
        result = college_connexion.execute(text("""
DROP TABLE IF EXISTS parcours
        """))
        display(result)
        transaction.commit()
    except:
        transaction.rollback()
        raise


In [ ]:
with college_connexion.begin() as transaction:
    try:
        result = college_connexion.execute(text("""
CREATE TABLE parcours (
    id INTEGER NOT NULL, 
    name VARCHAR,
    PRIMARY KEY (id)
)
"""))
        display(result)
        transaction.commit()
    except:
        transaction.rollback()
        raise

# 3. SQL Magic (notebook)